# ETL Pipeline: Mobile Gaming Platform (Google Play Store)
**Mục tiêu:** Tiền xử lý, làm sạch và chuẩn hóa dữ liệu Mobile từ `data/raw/mobile_playstore_games_2000+.csv`.

### Các bước xử lý chính:
1. Đọc và xử lý trường `realInstalls` làm chỉ số Volume.
2. Làm sạch thông tin thể loại, cờ F2P (`is_free`) và cờ nạp thẻ (`offersIAP`).
3. Quy đổi điểm số đánh giá 0 - 5 sao sang thang điểm chuẩn 0 - 100.
4. Tính doanh thu ước tính kết hợp ARPU thể loại từ dữ liệu vĩ mô.

## Bước 1: Chuẩn bị thư viện và đường dẫn

Nạp các thư viện cần thiết, khai báo đường dẫn tới dữ liệu Mobile, bảng doanh thu theo genre, bảng doanh thu khu vực và các file đầu ra.

In [1]:
import pandas as pd
import numpy as np
import os

# Đường dẫn dữ liệu
RAW_DATA_PATH = os.path.join('..', '..', 'data', 'raw', 'mobile_playstore_games_10000+.csv')
ARPU_PATH = os.path.join('..', '..', 'data', 'processed', 'genre_arpu_benchmark.csv')
REGION_REV_PATH = os.path.join('..', '..', 'data', 'external', 'revenue_by_region(2021 - 2025).csv')
PROCESSED_REGION_PATH = os.path.join('..', '..', 'data', 'processed', 'revenue_by_region_cleaned.csv')
PROCESSED_DATA_PATH = os.path.join('..', '..', 'data', 'processed', 'mobile_cleaned.csv')

print(f"Raw data path: {RAW_DATA_PATH}")

Raw data path: ..\..\data\raw\mobile_playstore_games_10000+.csv


## Bước 2: Đọc và lọc dữ liệu

Đọc dữ liệu Mobile cùng benchmark doanh thu theo genre. Loại game có rating bằng 0 hoặc thiếu năm phát hành để tránh đưa các bản ghi này vào phân tích rating và thời gian.

In [2]:
df_mobile = pd.read_csv(RAW_DATA_PATH)
df_genre_rev = pd.read_csv(ARPU_PATH)

# Loại game có rating bằng 0 hoặc thiếu/năm phát hành bằng 0 trước ETL.
df_mobile["score"] = pd.to_numeric(df_mobile["score"], errors="coerce")
df_mobile["release_year"] = pd.to_numeric(df_mobile["release_year"], errors="coerce")
rows_before_filter = len(df_mobile)
df_mobile = df_mobile.loc[
    df_mobile["score"].gt(0)
    & df_mobile["release_year"].notna()
    & df_mobile["release_year"].gt(0)
].copy()
print(f"Đã loại {rows_before_filter - len(df_mobile)} dòng có rating = 0 hoặc thiếu năm phát hành")
print(f"Số dòng còn lại: {len(df_mobile)}")

df_mobile.head(3)

Đã loại 1930 dòng có rating = 0 hoặc thiếu năm phát hành
Số dòng còn lại: 13070


,app_id,title,genre,genre_id,realInstalls,min_installs,installs_range,score,ratings,reviews,...,offersIAP,inAppProductPrice,developer,developer_id,released,release_year,version,content_rating,description,playstore_url
0,com.alexanderwinn.landfall,TerraGenesis: Landfall,Simulation,GAME_SIMULATION,1733498,1000000,"1,000,000+",4.07,26086,2006,...,True,$0.49 - $99.99 per item,Tilting Point,5479878731323183535,"Aug 9, 2022",2022.0,2.92,Everyone,From the makers of TerraGenesis comes TERRAGEN...,https://play.google.com/store/apps/details?id=...
1,com.rsgapps.reactor,Reactor - Energy Sector Tycoon,Strategy,GAME_STRATEGY,9415262,5000000,"5,000,000+",4.44,573229,15419,...,True,$0.99 - $9.99 per item,RSGapps - Idle Tycoon Games,6957790963724025053,"Mar 4, 2016",2016.0,1.72.65,Everyone,Become an industrial idle tycoon by managing y...,https://play.google.com/store/apps/details?id=...
2,com.tankionline.mobile.production,Tanki Online: PvP Tank Battle,Arcade,GAME_ARCADE,4830368,1000000,"1,000,000+",3.35,157853,1612,...,True,$4.99 - $248.99 per item,Alternativa Games,9219768540245481297,"Oct 7, 2018",2018.0,2.6.0.2002432142,Everyone 10+,Dive into the exhilarating world of Tanki Onli...,https://play.google.com/store/apps/details?id=...


## Bước 3: Chuẩn hóa lượt cài, genre và thông tin giá

Dùng realInstalls làm Volume_Units, ánh xạ genre nguồn sang taxonomy benchmark, đồng thời làm sạch các cờ miễn phí, mua hàng trong ứng dụng và giá game.

In [3]:
# Gán realInstalls làm Volume_Units.
df_mobile['Volume_Units'] = df_mobile['realInstalls'].fillna(0).astype(float)

# Chuẩn hóa genre theo taxonomy của bảng benchmark.
genre_mapping = {
    'Role Playing': 'RPG',
    'Board': 'Tabletop',
    'Card': 'Tabletop',
    'Casual': 'Puzzle',
    'Word': 'Puzzle',
    'Trivia': 'Puzzle',
    'Educational': 'Puzzle',
    'Music': 'Arcade',
    'Adventure': 'Action'
}
df_mobile['Genre_Clean'] = df_mobile['genre'].replace(genre_mapping)

# Làm sạch cờ game miễn phí và mua hàng trong ứng dụng.
df_mobile['is_free'] = df_mobile['is_free'].fillna(True).astype(bool)
df_mobile['offersIAP'] = df_mobile['offersIAP'].fillna(False).astype(bool)
df_mobile['price_usd'] = df_mobile['price_usd'].fillna(0.0)

df_mobile[['title', 'Volume_Units', 'Genre_Clean', 'is_free', 'offersIAP']].head()

,title,Volume_Units,Genre_Clean,is_free,offersIAP
0,TerraGenesis: Landfall,1733498.0,Simulation,True,True
1,Reactor - Energy Sector Tycoon,9415262.0,Strategy,True,True
2,Tanki Online: PvP Tank Battle,4830368.0,Arcade,True,True
3,Golf Blitz,3181690.0,Sports,True,True
4,Shadow Fight 3 - RPG fighting,210874943.0,RPG,True,True


## Bước 4: Chuẩn hóa rating và phân hạng

Chuyển điểm Google Play từ thang 0–5 sang thang 0–100, sau đó tạo Rating_Tier theo các ngưỡng đã thống nhất trong kế hoạch ETL.

In [4]:
# Chuyển điểm 0-5 sao sang thang 0-100.
df_mobile['Rating'] = df_mobile['score']

def assign_rating_tier(row):
      score = row.get('score', 0)
      ratings = row.get('ratings', 0)

      if score >= 4.5:
            if pd.notna(ratings) and ratings >= 100:
                  return 'Masterpiece'
            else:
                  return 'Great'
      elif score >= 4.0:
            return 'Great'
      elif score >= 3.5:
            return 'Mixed'
      else:
            return 'Poor'

df_mobile['Rating_Tier'] = df_mobile.apply(assign_rating_tier, axis = 1)
df_mobile[['title', 'score', 'Rating', 'Rating_Tier']].head()

,title,score,Rating,Rating_Tier
0,TerraGenesis: Landfall,4.07,4.07,Great
1,Reactor - Energy Sector Tycoon,4.44,4.44,Great
2,Tanki Online: PvP Tank Battle,3.35,3.35,Poor
3,Golf Blitz,4.27,4.27,Great
4,Shadow Fight 3 - RPG fighting,4.59,4.59,Masterpiece


## Bước 5: Ước tính ARPU và doanh thu

Tính ARPU theo genre từ doanh thu benchmark năm 2024 và tổng lượt cài trong dữ liệu Mobile. Dùng ARPU cùng lượt cài và giá game để tạo Estimated_Revenue_USD; đây là giá trị ước tính theo benchmark.

In [5]:
# Tính ARPU theo genre = doanh thu benchmark chia tổng lượt cài.
genre_rev_lookup = df_genre_rev.set_index('Genre')['Revenue_per_Download_Mobile'].to_dict()

df_mobile['ARPU'] = df_mobile['Genre_Clean'].map(genre_rev_lookup).fillna(0)
df_mobile['ARPU'] = df_mobile['ARPU'].fillna(df_genre_rev['Revenue_per_Download_Mobile'].median())
df_mobile['IAP_Revenue_USD'] = np.where(
    df_mobile['offersIAP'].eq(True),
    df_mobile['realInstalls'] * df_mobile['ARPU'].fillna(0),
    0.0
)
df_mobile['Paid_Revenue_USD'] = (df_mobile['price_usd'] * df_mobile['realInstalls']) 

# Ước tính doanh thu chuẩn hóa từ lượt cài, ARPU genre và giá game.
df_mobile['Estimated_Revenue_USD'] =  df_mobile['IAP_Revenue_USD'] + df_mobile['Paid_Revenue_USD']

# Tinh % tich luy theo developer
dev_df = df_mobile.groupby('developer')['Estimated_Revenue_USD'].sum().reset_index()
dev_df = dev_df.sort_values(by='Estimated_Revenue_USD', ascending = False)
dev_df['Cumulative_Revenue'] = dev_df['Estimated_Revenue_USD'].cumsum()

total_market_revenue = dev_df['Estimated_Revenue_USD'].sum()
dev_df['Cumulative_Revenue_Pct'] = (dev_df['Cumulative_Revenue'] / total_market_revenue) * 100

dev_cum_dict = dev_df.set_index('developer')['Cumulative_Revenue_Pct'].to_dict()
df_mobile['Cumulative_Revenue_Pct'] = df_mobile['developer'].map(dev_cum_dict)

df_mobile[['title', 'realInstalls', 'ARPU', 'Estimated_Revenue_USD']].head()
df_mobile

,app_id,title,genre,genre_id,realInstalls,min_installs,installs_range,score,ratings,reviews,...,playstore_url,Volume_Units,Genre_Clean,Rating,Rating_Tier,ARPU,IAP_Revenue_USD,Paid_Revenue_USD,Estimated_Revenue_USD,Cumulative_Revenue_Pct
0,com.alexanderwinn.landfall,TerraGenesis: Landfall,Simulation,GAME_SIMULATION,1733498,1000000,"1,000,000+",4.07,26086,2006,...,https://play.google.com/store/apps/details?id=...,1733498.0,Simulation,4.07,Great,0.56,9.707589e+05,0.0,9.707589e+05,62.105794
1,com.rsgapps.reactor,Reactor - Energy Sector Tycoon,Strategy,GAME_STRATEGY,9415262,5000000,"5,000,000+",4.44,573229,15419,...,https://play.google.com/store/apps/details?id=...,9415262.0,Strategy,4.44,Great,9.62,9.057482e+07,0.0,9.057482e+07,86.404755
2,com.tankionline.mobile.production,Tanki Online: PvP Tank Battle,Arcade,GAME_ARCADE,4830368,1000000,"1,000,000+",3.35,157853,1612,...,https://play.google.com/store/apps/details?id=...,4830368.0,Arcade,3.35,Poor,0.14,6.762515e+05,0.0,6.762515e+05,99.921613
3,com.noodlecake.ssg4,Golf Blitz,Sports,GAME_SPORTS,3181690,1000000,"1,000,000+",4.27,55068,6656,...,https://play.google.com/store/apps/details?id=...,3181690.0,Sports,4.27,Great,1.35,4.295282e+06,0.0,4.295282e+06,80.714524
4,com.nekki.shadowfight3,Shadow Fight 3 - RPG fighting,Role Playing,GAME_ROLE_PLAYING,210874943,100000000,"100,000,000+",4.59,4513435,75403,...,https://play.google.com/store/apps/details?id=...,210874943.0,RPG,4.59,Masterpiece,10.62,2.239492e+09,0.0,2.239492e+09,22.253172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,com.lunosoft.escaperoom,Escape Room Universe: Survival,Adventure,GAME_ADVENTURE,150875,100000,"100,000+",3.44,686,30,...,https://play.google.com/store/apps/details?id=...,150875.0,Action,3.44,Poor,1.26,1.901025e+05,0.0,1.901025e+05,99.855305
14996,wc.welcomecenter.octopus,Octopus Escape: Mind Control,Casual,GAME_CASUAL,3667123,1000000,"1,000,000+",4.44,11745,27,...,https://play.google.com/store/apps/details?id=...,3667123.0,Puzzle,4.44,Great,1.44,0.000000e+00,0.0,0.000000e+00,78.245441
14997,com.fl.paint.hide.seek.challenge,Chameleon Escape: Hide & Seek,Role Playing,GAME_ROLE_PLAYING,782014,500000,"500,000+",4.51,2908,9,...,https://play.google.com/store/apps/details?id=...,782014.0,RPG,4.51,Masterpiece,10.62,0.000000e+00,0.0,0.000000e+00,100.000000
14998,com.hidea.leaguepuzzle,Match Clash: 3 Puzzle Game,Puzzle,GAME_PUZZLE,32037,10000,"10,000+",3.89,620,33,...,https://play.google.com/store/apps/details?id=...,32037.0,Puzzle,3.89,Mixed,1.44,4.613328e+04,0.0,4.613328e+04,36.857876


## Bước 6: Tạo và lưu bảng Mobile theo schema master

Chuẩn hóa các trường về schema 12 cột dùng chung. Lưu kết quả vào data/processed/mobile_cleaned.csv để dùng ở bước ghép master.

In [6]:
clean_mobile = pd.DataFrame({
    'Game_Title': df_mobile['title'].astype(str).str.strip(),
    'Platform_Category': 'Mobile',
    'Genre': df_mobile['Genre_Clean'],
    'publisher': df_mobile['developer'].fillna('Unknown'),
    'developer': df_mobile['developer'].fillna('Unknown'),
    'release_year': df_mobile['release_year'].astype(int),
    'Volume_Units': df_mobile['Volume_Units'],
    'IAP_Revenue_USD' : df_mobile['IAP_Revenue_USD'],
    'Paid_Revenue_USD' : df_mobile['Paid_Revenue_USD'],
    'Estimated_Revenue_USD': df_mobile['Estimated_Revenue_USD'],
    'Rating': df_mobile['Rating'],
    'Rating_Tier': df_mobile['Rating_Tier'],
    'isFree' : df_mobile['is_free'],
    'offers_IAP' : df_mobile['offersIAP']
})

os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
clean_mobile.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Đã lưu thành công {len(clean_mobile)} dòng vào: {PROCESSED_DATA_PATH}")
clean_mobile.head()

Đã lưu thành công 13070 dòng vào: ..\..\data\processed\mobile_cleaned.csv


,Game_Title,Platform_Category,Genre,publisher,developer,release_year,Volume_Units,IAP_Revenue_USD,Paid_Revenue_USD,Estimated_Revenue_USD,Rating,Rating_Tier,isFree,offers_IAP
0,TerraGenesis: Landfall,Mobile,Simulation,Tilting Point,Tilting Point,2022,1733498.0,9.707589e+05,0.0,9.707589e+05,4.07,Great,True,True
1,Reactor - Energy Sector Tycoon,Mobile,Strategy,RSGapps - Idle Tycoon Games,RSGapps - Idle Tycoon Games,2016,9415262.0,9.057482e+07,0.0,9.057482e+07,4.44,Great,True,True
2,Tanki Online: PvP Tank Battle,Mobile,Arcade,Alternativa Games,Alternativa Games,2018,4830368.0,6.762515e+05,0.0,6.762515e+05,3.35,Poor,True,True
3,Golf Blitz,Mobile,Sports,Noodlecake,Noodlecake,2019,3181690.0,4.295282e+06,0.0,4.295282e+06,4.27,Great,True,True
4,Shadow Fight 3 - RPG fighting,Mobile,RPG,NEKKI,NEKKI,2017,210874943.0,2.239492e+09,0.0,2.239492e+09,4.59,Masterpiece,True,True


## Bước 7: Làm sạch bảng doanh thu theo khu vực

Đọc CSV với keep_default_na=False để giữ mã vùng NA, chuẩn hóa kiểu dữ liệu, điền 0 cho tăng trưởng YoY năm cơ sở 2021 và lưu file đã làm sạch.

In [7]:
# 1. Dùng keep_default_na=False để giữ nguyên mã vùng NA.
df_region_raw = pd.read_csv(REGION_REV_PATH, keep_default_na=False)

# 2. Chuẩn hóa kiểu dữ liệu và xử lý giá trị trống.
df_region = df_region_raw.copy()
df_region["Region"] = df_region["Region"].astype(str).str.strip()
df_region["Region_Code"] = df_region["Region_Code"].astype(str).str.strip()
df_region["Year"] = df_region["Year"].astype(int)
df_region["Share_Pct"] = pd.to_numeric(
    df_region["Share_Pct"], errors="coerce"
).fillna(0.0)

# Năm cơ sở 2021 không có tăng trưởng YoY; điền 0.0.
df_region["YoY_Growth_Pct"] = pd.to_numeric(
    df_region["YoY_Growth_Pct"], errors="coerce"
).fillna(0.0)

# 3. Kiểm tra mã vùng NA được giữ nguyên.
print(
    f"Mã vùng NA còn nguyên vẹn: {('NA' in df_region['Region_Code'].values)}"
)
print(f"Số lượng missing value:\n{df_region.isnull().sum()}")

# 4. Lưu bảng đã làm sạch vào thư mục processed.
os.makedirs(os.path.dirname(PROCESSED_REGION_PATH), exist_ok=True)
df_region.to_csv(PROCESSED_REGION_PATH, index=False)

print(f"Đã lưu thành công {len(df_region)} dòng vào: {PROCESSED_REGION_PATH}")
df_region.head(6)

Mã vùng NA còn nguyên vẹn: True
Số lượng missing value:
Region            0
Region_Code       0
Year              0
Share_Pct         0
YoY_Growth_Pct    0
dtype: int64
Đã lưu thành công 25 dòng vào: ..\..\data\processed\revenue_by_region_cleaned.csv


,Region,Region_Code,Year,Share_Pct,YoY_Growth_Pct
0,Asia-Pacific,APAC,2021,48.3,0.0
1,North America,NA,2021,26.5,0.0
2,Europe,EU,2021,17.7,0.0
3,Latin America,LATAM,2021,4.2,0.0
4,Middle East & Africa,MENA,2021,3.3,0.0
5,Asia-Pacific,APAC,2022,47.7,-5.6


## Bước 8: Xuất biểu đồ EDA tĩnh cho Mobile

Tạo năm biểu đồ phân phối, tương quan, lượt cài theo genre, số game theo genre và rating so với doanh thu. Ảnh được lưu trong visuals/eda_static/mobile/; boxplot dùng genre vì hiện chỉ phân tích nền tảng Mobile.

In [8]:
# EDA Mobile: lưu 5 biểu đồ tĩnh vào visuals/eda_static/mobile.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = (project_root / ".." / "..").resolve()
processed_path = project_root / "data" / "processed" / "mobile_cleaned.csv"
output_dir = project_root / "visuals" / "eda_static" / "mobile"
output_dir.mkdir(parents=True, exist_ok=True)

df_eda = pd.read_csv(processed_path)
df_eda["Volume_Units"] = pd.to_numeric(df_eda["Volume_Units"], errors="coerce")
df_eda["Estimated_Revenue_USD"] = pd.to_numeric(df_eda["Estimated_Revenue_USD"], errors="coerce")
df_eda["Rating"] = pd.to_numeric(df_eda["Rating"], errors="coerce")
df_eda["release_year"] = pd.to_numeric(df_eda["release_year"], errors="coerce")
sns.set_theme(style="whitegrid", context="notebook", palette="Set2")

# 1. Phân phối doanh thu: histogram và KDE trên log10 doanh thu USD.
revenue = df_eda.loc[df_eda["Estimated_Revenue_USD"] > 0, "Estimated_Revenue_USD"].dropna()
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(np.log10(revenue), bins=35, kde=True, color="#4472C4", ax=ax)
# 1. Phân phối doanh thu
ax.set(
    title="Mobile: Phân phối doanh thu ước tính",
    xlabel="log10(Doanh thu ước tính, USD)",
    ylabel="Số lượng game"
)
fig.tight_layout()
fig.savefig(output_dir / "eda_1_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 2. Heatmap tương quan; log1p giúp giảm độ lệch của lượt cài và doanh thu.
corr_data = pd.DataFrame({
    "Release year": df_eda["release_year"],
    "log1p(Volume)": np.log1p(df_eda["Volume_Units"].clip(lower=0)),
    "log1p(Revenue)": np.log1p(df_eda["Estimated_Revenue_USD"].clip(lower=0)),
    "Rating / 100": df_eda["Rating"],
}).dropna()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data.corr(), annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, square=True, ax=ax)
# 2. Ma trận tương quan
ax.set_title("Mobile: Ma trận tương quan giữa các biến số")
fig.tight_layout()
fig.savefig(output_dir / "eda_2_correlation.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 3. Boxplot lượt cài theo genre trong tập Mobile.
box_data = df_eda.loc[(df_eda["Volume_Units"] > 0) & df_eda["Genre"].notna()].copy()
box_data["log10_Volume"] = np.log10(box_data["Volume_Units"])
genre_order = box_data.groupby("Genre")["Volume_Units"].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 7))
sns.boxplot(data=box_data, x="log10_Volume", y="Genre", order=genre_order, showfliers=False, color="#8FC7B5", ax=ax)
# 3. Phân phối lượt cài đặt theo thể loại
ax.set(
    title="Mobile: Phân phối lượt cài đặt theo thể loại",
    xlabel="log10(Số lượt cài đặt)",
    ylabel="Thể loại"
)
fig.tight_layout()
fig.savefig(output_dir / "eda_3_volume_by_genre.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 4. Số lượng game theo genre.
fig, ax = plt.subplots(figsize=(11, 6))
order = df_eda["Genre"].value_counts().index
sns.countplot(data=df_eda, y="Genre", order=order, color="#70AD47", ax=ax)
# 4. Số lượng game theo thể loại
ax.set(
    title="Mobile: Số lượng game theo thể loại",
    xlabel="Số lượng game",
    ylabel="Thể loại"
)
for container in ax.containers:
    ax.bar_label(container, padding=3, fontsize=8)
fig.tight_layout()
fig.savefig(output_dir / "eda_4_genre_count.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# 5. Rating so với doanh thu, trục doanh thu dùng log10.
scatter_data = df_eda.loc[(df_eda["Estimated_Revenue_USD"] > 0) & df_eda["Rating"].notna()].copy()
scatter_data["log10_Revenue"] = np.log10(scatter_data["Estimated_Revenue_USD"])
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=scatter_data, x="Rating", y="log10_Revenue", hue="Genre", alpha=0.65, s=35, palette="tab10", ax=ax)
# 5. Điểm đánh giá và doanh thu
ax.set(
    title="Mobile: Mối quan hệ giữa điểm đánh giá và doanh thu ước tính",
    xlabel="Điểm đánh giá chuẩn hóa (0–100)",
    ylabel="log10(Doanh thu ước tính, USD)"
)

ax.legend(
    title="Thể loại",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)
fig.tight_layout()
fig.savefig(output_dir / "eda_5_scatter.png", dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved 5 Mobile EDA charts to: {output_dir}")


Saved 5 Mobile EDA charts to: D:\Documents\TuongTacDuLieu\GAMING INDUSTRY ANALYSIS\Gaming-Market-Industry-Analytics\visuals\eda_static\mobile
